Jab bhi koi function DataFrame ko restructure karta hai (pivot, pivot_table, set_index, groupby), result ko NAYE variable mein store karo, df ko overwrite mat karo — jab tak tumhe original df baad mein zarurat na ho.

## pivot() — Sirf Reshape Karna
- df.pivot(index='col1', columns='col2', values='col3')
- Long format ko wide format mein badalta hai (rows ko columns mein convert)
- values=[...] (list) diya toh multiple columns ke liye reshape hoga
- LIMITATION: Agar index+columns ka combination DUPLICATE ho, error aayega
  (Pandas confuse ho jaata hai konsi value rakhe)

In [22]:
import pandas as pd
df = pd.read_csv("datasets/weather1.csv")
df

,date,city,temperature,humidity
0,5/1/2017,new york,65,56
1,5/2/2017,new york,66,58
2,5/3/2017,new york,68,60
3,5/1/2017,mumbai,75,80
4,5/2/2017,mumbai,78,83
5,5/3/2017,mumbai,82,85
6,5/1/2017,beijing,80,26
7,5/2/2017,beijing,77,30
8,5/3/2017,beijing,79,35


In [23]:
pivoted_multi = df.pivot(index="date", columns="city", values=["temperature", "humidity"])
pivoted_multi

temperature                 humidity                
city         beijing mumbai new york  beijing mumbai new york
date                                                         
5/1/2017          80     75       65       26     80       56
5/2/2017          77     78       66       30     83       58
5/3/2017          79     82       68       35     85       60

In [24]:
pivoted_temp = df.pivot(index="date", columns="city", values="temperature")
pivoted_temp

city,beijing,mumbai,new york
date,,,
5/1/2017,80,75,65
5/2/2017,77,78,66
5/3/2017,79,82,68


## pivot_table() — Reshape + Aggregate (Summarize)
- Jaisa groupby(), but table/grid format mein result deta hai
- aggfunc parameter se duplicates ko combine karta hai (default aggfunc="mean")
- Duplicate combinations pe error NAHI deta (pivot() ke ulat)

In [25]:
df1 = pd.read_csv("datasets/weather2.csv")
df1

,date,city,temperature,humidity
0,5/1/2017,new york,65,56
1,5/1/2017,new york,61,54
2,5/2/2017,new york,70,60
3,5/2/2017,new york,72,62
4,5/1/2017,mumbai,75,80
5,5/1/2017,mumbai,78,83
6,5/2/2017,mumbai,82,85
7,5/2/2017,mumbai,80,26


In [ ]:
df1 = df1.pivot_table(index="city",columns="date")
df1

humidity          temperature         
date     5/1/2017 5/2/2017    5/1/2017 5/2/2017
city                                           
mumbai       81.5     55.5        76.5     81.0
new york     55.0     61.0        63.0     71.0

In [27]:
df1 = df1.pivot_table(index="city",columns="date", aggfunc="sum")
df1

humidity          temperature         
date 5/1/2017 5/2/2017    5/1/2017 5/2/2017
city     81.5     55.5        76.5     81.0
date     55.0     61.0        63.0     71.0

In [28]:
df1 = df1.pivot_table(
    index="city",
    columns="date",
    values="temperature",
    aggfunc="sum"
)
df1

date,5/1/2017,5/2/2017
city,76.5,81.0
date,63.0,71.0


## Grouper() — Time-Based Grouping (Monthly/Weekly/etc.)
- pd.Grouper(freq='M', key='date_column') → date column ko month-wise group karta hai
- freq values: 'D'=Daily, 'W'=Weekly, 'M'=Monthly, 'Y'=Yearly
- Real use: daily data ko monthly summary mein convert karna
- IMPORTANT: date column pehle pd.to_datetime() se convert hona chahiye
- NAYA VARIABLE USE KARO result store karne ke liye (df ko overwrite mat karo),
  warna dobara run karne pe original column "date"/"city" nahi milenge (KeyError)

In [29]:
df2 = pd.read_csv("datasets/weather3.csv")
df2

,date,city,temperature,humidity
0,5/1/2017,new york,65,56
1,5/2/2017,new york,61,54
2,5/3/2017,new york,70,60
3,12/1/2017,new york,30,50
4,12/2/2017,new york,28,52
5,12/3/2017,new york,25,51


In [30]:
df2['date']=pd.to_datetime(df2["date"])

In [ ]:
df3 = df2.pivot_table(index=pd.Grouper(freq='M',key='date'),columns='city')
df3

C:\Users\ap504\AppData\Local\Temp\ipykernel_21128\3404683152.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df3 = df2.pivot_table(index=pd.Grouper(freq='M',key='date'),columns='city')


,humidity,temperature
city,new york,new york
date,,
2017-05-31,56.666667,65.333333
2017-12-31,51.000000,27.666667


: 